# PS6E4: 30-Model Ensemble with Stacking

**Competition:** [Playground Series S6E4](https://www.kaggle.com/competitions/playground-series-s6e4)  
**Task:** 3-class classification (Low, Medium, High) of irrigation need  
**Metric:** Balanced Accuracy  

This notebook covers:
1. **EDA** of the competition data
2. **Workflow overview**: how models were trained on AWS SageMaker
3. **Ensemble** from pre-computed OOF predictions using greedy selection, LightGBM stacking, rank averaging, and threshold optimization

All 30 models were trained on AWS SageMaker (CPU and GPU instances). This notebook only runs the ensembling step, which is CPU-only.

## 1. Setup

In [ ]:
import warnings
from pathlib import Path

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from scipy.optimize import differential_evolution
from scipy.stats import rankdata

from sklearn.metrics import balanced_accuracy_score, confusion_matrix, ConfusionMatrixDisplay
from sklearn.model_selection import StratifiedKFold
from sklearn.preprocessing import LabelEncoder
from sklearn.linear_model import LogisticRegression

import lightgbm as lgb

warnings.filterwarnings("ignore")

sns.set_theme(
    style="whitegrid",
    palette="muted",
    font_scale=1.1,
    rc={"figure.figsize": (12, 6), "axes.titlesize": 14, "axes.labelsize": 12},
)

CLASS_COLORS = {"Low": "#27ae60", "Medium": "#f39c12", "High": "#e74c3c"}
CLASS_ORDER = ["Low", "Medium", "High"]

COMP_DIR = Path("/kaggle/input/competitions/playground-series-s6e4")
PRED_DIR = Path("/kaggle/input/datasets/wguesdon/ps6e4-30model-oof-predictions")
CODE_DIR = Path("/kaggle/input/datasets/wguesdon/ps6e4-training-code-30")

train = pd.read_csv(COMP_DIR / "train.csv")
test = pd.read_csv(COMP_DIR / "test.csv")
original = pd.read_csv(PRED_DIR / "irrigation_prediction.csv")

le = LabelEncoder()
le.fit(["High", "Low", "Medium"])  # alphabetical: High=0, Low=1, Medium=2
y_train = le.transform(train["Irrigation_Need"])

print(f"Train: {train.shape}")
print(f"Test:  {test.shape}")
print(f"Original dataset: {original.shape}")
print(f"Label encoding: {dict(zip(le.classes_, le.transform(le.classes_)))}")

## 2. Exploratory Data Analysis

### 2.1 Target Distribution

The target is imbalanced: Low dominates at ~59%, Medium is ~38%, and High is only ~3.3%. Since the metric is **balanced accuracy**, each class contributes equally regardless of size. Getting the minority High class right is critical.

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# Target distribution
counts = train["Irrigation_Need"].value_counts().reindex(CLASS_ORDER)
colors = [CLASS_COLORS[c] for c in CLASS_ORDER]
axes[0].bar(CLASS_ORDER, counts.values, color=colors, edgecolor="black", linewidth=0.5)
for i, (cls, cnt) in enumerate(zip(CLASS_ORDER, counts.values)):
    axes[0].text(i, cnt + 2000, f"{cnt:,}\n({cnt/len(train)*100:.1f}%)", ha="center", fontsize=11)
axes[0].set_title("Target Distribution (Train)")
axes[0].set_ylabel("Count")

# Compare train vs original
target_col = "Irrigation_Need" if "Irrigation_Need" in original.columns else original.columns[-1]
train_pct = train["Irrigation_Need"].value_counts(normalize=True).reindex(CLASS_ORDER)
orig_pct = original[target_col].value_counts(normalize=True).reindex(CLASS_ORDER)
x = np.arange(len(CLASS_ORDER))
w = 0.35
axes[1].bar(x - w/2, train_pct.values * 100, w, label="Synthetic (630K)", color="#3498db", edgecolor="black", linewidth=0.5)
axes[1].bar(x + w/2, orig_pct.values * 100, w, label="Original (10K)", color="#e67e22", edgecolor="black", linewidth=0.5)
axes[1].set_xticks(x)
axes[1].set_xticklabels(CLASS_ORDER)
axes[1].set_ylabel("Percentage")
axes[1].set_title("Train vs Original Dataset")
axes[1].legend()

plt.tight_layout()
plt.show()


### 2.2 Feature Overview

The dataset has 20 features: 11 numeric and 8 categorical (plus the id column).

In [ ]:
cats = [c for c in train.columns if train[c].dtype == "object" and c not in ["id", "Irrigation_Need"]]
nums = [c for c in train.columns if c not in cats and c not in ["id", "Irrigation_Need"]]

print(f"Numeric features ({len(nums)}): {nums}")
print(f"Categorical features ({len(cats)}): {cats}")
print(f"\nMissing values: {train.isnull().sum().sum()}")
print(f"\nNumeric summary:")
train[nums].describe().round(2)

In [ ]:
# Categorical cardinality
print("Categorical cardinality:")
for c in cats:
    print(f"  {c}: {train[c].nunique()} unique values")
    print(f"    {train[c].value_counts().head(5).to_dict()}")

### 2.3 Numeric Feature Distributions by Class

In [ ]:
fig, axes = plt.subplots(3, 4, figsize=(20, 12))
axes = axes.flatten()

sample = train.sample(50000, random_state=42)

for i, col in enumerate(nums):
    ax = axes[i]
    for cls in CLASS_ORDER:
        data = sample.loc[sample["Irrigation_Need"] == cls, col]
        ax.hist(data, bins=50, alpha=0.5, label=cls, color=CLASS_COLORS[cls], density=True)
    ax.set_title(col, fontsize=10)
    if i == 0:
        ax.legend(fontsize=8)

# Hide extra subplot
for j in range(len(nums), len(axes)):
    axes[j].set_visible(False)

plt.suptitle("Numeric Feature Distributions by Class (50K sample)", fontsize=14, y=1.01)
plt.tight_layout()
plt.show()

### 2.4 Correlation Matrix

In [ ]:
fig, ax = plt.subplots(figsize=(10, 8))
corr = train[nums].corr()
mask = np.triu(np.ones_like(corr, dtype=bool))
sns.heatmap(corr, mask=mask, annot=True, fmt=".2f", cmap="coolwarm", center=0,
            linewidths=0.5, ax=ax, vmin=-1, vmax=1)
ax.set_title("Numeric Feature Correlation")
plt.tight_layout()
plt.show()

## 3. Workflow Overview

### Architecture

All 30 models were trained on **AWS SageMaker** using spot and on-demand instances. The training code is attached as a separate Kaggle dataset (`ps6e4-training-code`). Here is how the pipeline works:

```
Local machine                    AWS SageMaker
=============                    =============
                                 
scripts/         ──upload──>     S3 bucket
  train_xgb_ote.py               s3://kaggle-ps6e4/
  train_cat_ote.py                  ├── ensemble-data/
  train_lgb_ote.py                  │   ├── train.csv
  ensemble_v4.py                    │   ├── test.csv
  ...                               │   └── predictions/
                                    │       ├── oof_*.npy
                                    │       └── pred_*.npy
Launch jobs via   ──API──>       Training Jobs
SageMaker SDK                      ml.c5.9xlarge (CPU, GBDTs)
                                   ml.g4dn.xlarge (GPU, NNs)
                                 
Download results  <──S3──        model.tar.gz
  oof_*.npy                        oof predictions (N_train, 3)
  pred_*.npy                       test predictions (N_test, 3)
```

Each model script:
1. Reads train/test CSV from the SageMaker input channel
2. Runs 5-fold StratifiedKFold (seed=42, consistent across all models)
3. Saves out-of-fold (OOF) predictions and test predictions as numpy arrays
4. Packages outputs into `model.tar.gz` uploaded to S3

### Two Feature Engineering Pipelines

**Pipeline 1: v3 features** (18 models: 7 CatBoost, 5 LightGBM, 6 XGBoost)
- Domain features: water balance, heat stress, evapotranspiration
- Digit extraction from numeric values
- Frequency encoding of categoricals
- Snapping/rounding for cardinality reduction
- Priors from the original 10K dataset (zero-leakage)
- Optuna hyperparameter tuning (50 trials per model)
- Script: `02_feature_engineering.py` + `03a_train_xgb.py` / `03b_train_lgb.py` / `03c_train_cat.py`

**Pipeline 2: Ordered Target Encoding (OTE)** (6 models, top performers)
- Based on [emanuellcs notebook](https://www.kaggle.com/code/emanuellcs/predicting-irrigation-need-xgboost-ote) (LB 0.98011)
- Bayesian-smoothed cumulative leave-one-out target encoding with 4x shuffle
- Digit extraction from numeric features (digits -4 to +3)
- Frequency-based ordinal encoding for categoricals and digit features
- XGBoost (3 seeds), CatBoost, LightGBM, and XGBoost+MagicFormula
- Script: `train_xgb_ote.py` / `train_cat_ote.py` / `train_lgb_ote.py`

**Other models** (3 models)
- RealMLP (2 variants): neural network using Mahogany features
- TabM: tabular deep learning model

### Why This Works

The two feature pipelines produce fundamentally different representations of the same data. v3 features are domain-driven (physics formulas, frequency counts). OTE features are target-driven (smoothed target statistics). When combined in a stacker, the model learns which pipeline is more informative for each region of the feature space. This complementarity is the main source of ensemble gain.

### 3.1 Model Inventory

| # | Model | Algorithm | Pipeline | CV |
|---|-------|-----------|----------|----|
| 1 | lgb_ote | LightGBM | OTE | 0.97942 |
| 2 | xgb_ote | XGBoost | OTE | 0.97938 |
| 3 | xgb_ote_s44 | XGBoost | OTE (seed 2044) | 0.97930 |
| 4 | xgb_ote_s43 | XGBoost | OTE (seed 2043) | 0.97920 |
| 5 | cat_ote | CatBoost | OTE | 0.97919 |
| 6 | xgb_ote_magic | XGBoost | OTE + magic | 0.97910 |
| 7 | cat_s44 | CatBoost | v3 | 0.97834 |
| 8 | cat_digit | CatBoost | v3 | 0.97828 |
| 9 | cat-...-13-29 | CatBoost | v3 | 0.97812 |
| 10 | cat-...-12-44-01 | CatBoost | v3 | 0.97804 |
| 11 | cat_s43 | CatBoost | v3 | 0.97802 |
| 12 | realmlp_mahog | RealMLP | mahogany | 0.97802 |
| 13 | cat-...-12-44-05 | CatBoost | v3 | 0.97801 |
| 14 | cat_v2 | CatBoost | v3 | 0.97786 |
| 15 | lgb_v5 | LightGBM | v3 | 0.97701 |
| 16 | lgb_digit | LightGBM | v3 | 0.97562 |
| 17 | xgb_v5 | XGBoost | v3 | 0.97561 |
| 18 | xgb-...-11-12-41 | XGBoost | v3 | 0.97262 |
| 19 | xgb-...-11-12-38 | XGBoost | v3 | 0.97256 |
| 20 | xgb_v2 | XGBoost | v3 | 0.97208 |
| 21 | xgb_s44 | XGBoost | v3 | 0.97199 |
| 22 | xgb_s43 | XGBoost | v3 | 0.97185 |
| 23 | lgb_s44 | LightGBM | v3 | 0.97171 |
| 24 | lgb_v2 | LightGBM | v3 | 0.97140 |
| 25 | lgb_s43 | LightGBM | v3 | 0.97131 |
| 26 | realmlp_v3fix | RealMLP | v3 | 0.97108 |
| 27 | tabm_v3fix | TabM | v3 | 0.97053 |
| 28 | lr_ote | Logistic Regression | OTE | 0.93743 |
| 29 | et_ote | ExtraTrees | OTE | 0.96156 |
| 30 | knn_ote | KNN | OTE | 0.77552 |

## 4. Load Predictions

In [ ]:
MODEL_NAMES = [
    # OTE pipeline (top tier)
    "lgb_ote", "xgb_ote", "xgb_ote_s44", "xgb_ote_s43", "cat_ote", "xgb_ote_magic",
    # v3 pipeline - CatBoost
    "cat_s44", "cat_digit", "cat-2026-04-03-13-29-13-994",
    "cat-2026-04-03-12-44-01-719", "cat_s43",
    "cat-2026-04-03-12-44-05-363", "cat_v2",
    # v3 pipeline - LightGBM
    "lgb_v5", "lgb_digit",
    # v3 pipeline - XGBoost
    "xgb_v5", "xgb-2026-04-03-11-12-41-698", "xgb-2026-04-03-11-12-38-123",
    "xgb_v2", "xgb_s44", "xgb_s43",
    # v3 pipeline - LightGBM (other seeds)
    "lgb_s44", "lgb_v2", "lgb_s43",
    # Neural networks
    "realmlp_mahog", "realmlp_v3fix", "tabm_v3fix",
    # Diversity models (weak individually, help stacker)
    "lr_ote", "knn_ote", "et_ote",
]

oof_dict = {}
pred_dict = {}

print(f"{'#':<4} {'Model':<42} {'CV (bal_acc)':>12}")
print("-" * 60)

for i, name in enumerate(MODEL_NAMES, 1):
    oof = np.load(PRED_DIR / f"oof_{name}.npy")
    pred = np.load(PRED_DIR / f"pred_{name}.npy")
    oof_dict[name] = oof
    pred_dict[name] = pred
    score = balanced_accuracy_score(y_train, oof.argmax(axis=1))
    print(f"{i:<4} {name:<42} {score:.5f}")

print(f"\nLoaded {len(MODEL_NAMES)} models.")
print(f"OOF shape: {oof.shape}, Test shape: {pred.shape}")

## 5. Model Diversity Analysis

For an ensemble to work, models need to make different errors. Let's examine how correlated the predictions are.

In [ ]:
oof_argmax = np.column_stack([oof_dict[m].argmax(axis=1) for m in MODEL_NAMES])
corr = np.corrcoef(oof_argmax.T)

short_names = []
for n in MODEL_NAMES:
    if len(n) > 15:
        short_names.append(n[:7] + ".." + n[-6:])
    else:
        short_names.append(n)

fig, ax = plt.subplots(figsize=(18, 16))
sns.heatmap(
    corr, xticklabels=short_names, yticklabels=short_names,
    annot=True, fmt=".2f", cmap="coolwarm", vmin=0.7, vmax=1.0,
    linewidths=0.3, ax=ax, annot_kws={"size": 7},
)
ax.set_title("OOF Prediction Correlation (argmax) across 27 Models")
plt.xticks(rotation=45, ha="right", fontsize=8)
plt.yticks(rotation=0, fontsize=8)
plt.tight_layout()
plt.show()

# Summary stats
mask = np.triu(np.ones_like(corr, dtype=bool), k=1)
upper = corr[mask]
print(f"\nPairwise correlation stats:")
print(f"  Mean: {upper.mean():.4f}")
print(f"  Min:  {upper.min():.4f}")
print(f"  Max:  {upper.max():.4f}")

### 5.1 Agreement Rate by Class

Where do models disagree the most? The minority High class should show the most disagreement.

In [ ]:
# For each sample, compute the fraction of models that agree on the prediction
from scipy.stats import mode

modal_pred = mode(oof_argmax, axis=1).mode.ravel()
agreement = np.mean(oof_argmax == modal_pred[:, None], axis=1)

fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# Agreement distribution
axes[0].hist(agreement, bins=50, color="steelblue", edgecolor="black", linewidth=0.5)
axes[0].axvline(agreement.mean(), color="red", linestyle="--", label=f"Mean: {agreement.mean():.3f}")
axes[0].set_xlabel("Fraction of models agreeing")
axes[0].set_ylabel("Number of samples")
axes[0].set_title("Model Agreement Distribution")
axes[0].legend()

# Agreement by true class
class_names = le.classes_  # High, Low, Medium
for cls_idx, cls_name in enumerate(class_names):
    mask_cls = y_train == cls_idx
    axes[1].hist(agreement[mask_cls], bins=30, alpha=0.5,
                 label=f"{cls_name} (mean: {agreement[mask_cls].mean():.3f})",
                 color=CLASS_COLORS[cls_name], density=True)
axes[1].set_xlabel("Fraction of models agreeing")
axes[1].set_ylabel("Density")
axes[1].set_title("Agreement by True Class")
axes[1].legend()

plt.tight_layout()
plt.show()

# Disagreement stats
high_disagree = (agreement < 0.6).sum()
print(f"Samples with <60% agreement: {high_disagree:,} ({high_disagree/len(agreement)*100:.2f}%)")

## 6. Ensemble Methods

We test five ensemble strategies and combine each with two post-processing methods (log-bias tuning and differential evolution thresholds). The LightGBM stacker on all 30 models is the best performing method.

### 6.1 Threshold Tuning Functions

In [ ]:
def tune_bias_log(proba, y_true):
    """Tune per-class additive biases in log-probability space.
    
    Multi-resolution grid search: starts with large steps,
    progressively refines. Adjusts decision boundaries without
    retraining.
    
    Args:
        proba: Class probability array (N, 3).
        y_true: Ground truth integer labels.
    
    Returns:
        Tuple of (best_bias array, best balanced accuracy).
    """
    def preds_from_bias(bias):
        return np.argmax(np.log(np.clip(proba, 1e-15, 1.0)) + bias, axis=1)

    best_bias = np.zeros(proba.shape[1], dtype=np.float64)
    best_score = balanced_accuracy_score(y_true, preds_from_bias(best_bias))

    for step in (1.0, 0.5, 0.2, 0.1, 0.05, 0.02, 0.01, 0.005):
        improved = True
        while improved:
            improved = False
            for ci in range(proba.shape[1]):
                for d in (-1.0, 1.0):
                    candidate = best_bias.copy()
                    candidate[ci] += d * step
                    s = balanced_accuracy_score(y_true, preds_from_bias(candidate))
                    if s > best_score + 1e-8:
                        best_bias = candidate
                        best_score = s
                        improved = True
    return best_bias, best_score


def diffevol_thresholds(proba, y_true):
    """Optimize multiplicative per-class thresholds via differential evolution.
    
    Args:
        proba: Class probability array (N, 3).
        y_true: Ground truth integer labels.
    
    Returns:
        Tuple of (best_thresholds array, best balanced accuracy).
    """
    def neg_ba(thresholds):
        return -balanced_accuracy_score(y_true, (proba * thresholds).argmax(axis=1))

    result = differential_evolution(
        neg_ba,
        bounds=[(0.3, 4.0), (0.3, 4.0), (0.3, 4.0)],
        seed=42, maxiter=2000, popsize=30, tol=1e-12,
    )
    return result.x, -result.fun


def apply_best_threshold(oof_proba, test_proba, y_true):
    """Try both threshold methods and return the better one.
    
    Args:
        oof_proba: OOF probability array.
        test_proba: Test probability array.
        y_true: Ground truth integer labels.
    
    Returns:
        Tuple of (test predictions, cv score, method name).
    """
    bias, score_bias = tune_bias_log(oof_proba, y_true)
    thresh, score_de = diffevol_thresholds(oof_proba, y_true)

    if score_bias >= score_de:
        test_preds = np.argmax(
            np.log(np.clip(test_proba, 1e-15, 1.0)) + bias, axis=1
        )
        return test_preds, score_bias, "log_bias"
    else:
        test_preds = (test_proba * thresh).argmax(axis=1)
        return test_preds, score_de, "diffevol"

### 6.2 Greedy Forward Selection

Start empty, iteratively add the model that improves ensemble CV the most. Stop when no model helps.

In [ ]:
def greedy_forward_selection(oof_dict, y_true, model_names):
    """Greedy forward model selection with log-bias tuning at each step.
    
    Args:
        oof_dict: Dict mapping model name to OOF predictions (N, 3).
        y_true: Ground truth integer labels.
        model_names: List of candidate model names.
    
    Returns:
        Tuple of (selected model names list, best CV score).
    """
    selected = []
    remaining = set(model_names)
    best_score = 0.0

    print(f"{'Step':<6} {'Added Model':<42} {'CV':>10} {'Delta':>10}")
    print("-" * 70)

    while remaining:
        best_add = None
        best_add_score = best_score

        for candidate in sorted(remaining):
            trial = selected + [candidate]
            blend = np.mean([oof_dict[n] for n in trial], axis=0)
            _, score = tune_bias_log(blend, y_true)
            if score > best_add_score + 1e-6:
                best_add_score = score
                best_add = candidate

        if best_add is None:
            break

        delta = best_add_score - best_score
        selected.append(best_add)
        remaining.remove(best_add)
        best_score = best_add_score
        delta_str = f"+{delta:.5f}" if len(selected) > 1 else "--"
        print(f"{len(selected):<6} {best_add:<42} {best_score:.5f} {delta_str:>10}")

    print(f"\nSelected {len(selected)} models, CV = {best_score:.5f}")
    return selected, best_score


selected, gfs_score = greedy_forward_selection(oof_dict, y_train, MODEL_NAMES)

In [ ]:
# Apply threshold tuning to greedy ensemble
greedy_oof = np.mean([oof_dict[n] for n in selected], axis=0)
greedy_test = np.mean([pred_dict[n] for n in selected], axis=0)
greedy_preds, greedy_cv, greedy_method = apply_best_threshold(greedy_oof, greedy_test, y_train)
print(f"Greedy ({len(selected)}m) + {greedy_method}: CV = {greedy_cv:.5f}")

### 6.3 LightGBM Stacker (best method)

Use all 30 models' OOF probabilities as features (90 features = 30 models x 3 classes). Train a LightGBM classifier with 5-fold CV. The stacker learns per-sample weights for each base model, which is more flexible than fixed-weight averaging.

In [ ]:
def lgb_stacker(oof_dict, pred_dict, y_true, model_names, n_splits=5, seed=42):
    """LightGBM stacker on OOF probabilities.
    
    Args:
        oof_dict: Dict mapping model name to OOF predictions.
        pred_dict: Dict mapping model name to test predictions.
        y_true: Ground truth integer labels.
        model_names: List of model names to use as features.
        n_splits: Number of CV folds.
        seed: Random state.
    
    Returns:
        Tuple of (oof_proba, test_proba, cv_score).
    """
    X_oof = np.hstack([oof_dict[n] for n in model_names])
    X_test = np.hstack([pred_dict[n] for n in model_names])

    n_classes = 3
    oof_proba = np.zeros((len(y_true), n_classes))
    test_probas = []

    skf = StratifiedKFold(n_splits=n_splits, shuffle=True, random_state=seed)

    params = {
        "objective": "multiclass",
        "num_class": n_classes,
        "metric": "multi_logloss",
        "learning_rate": 0.05,
        "num_leaves": 31,
        "min_child_samples": 50,
        "subsample": 0.8,
        "colsample_bytree": 0.8,
        "reg_alpha": 0.1,
        "reg_lambda": 1.0,
        "verbose": -1,
        "seed": seed,
    }

    print(f"Training LGB stacker on {X_oof.shape[1]} features ({len(model_names)} models x 3 classes)")
    fold_scores = []

    for fold, (tr_idx, va_idx) in enumerate(skf.split(X_oof, y_true)):
        dtrain = lgb.Dataset(X_oof[tr_idx], label=y_true[tr_idx])
        dval = lgb.Dataset(X_oof[va_idx], label=y_true[va_idx], reference=dtrain)

        model = lgb.train(
            params, dtrain, num_boost_round=1000,
            valid_sets=[dval],
            callbacks=[lgb.early_stopping(50), lgb.log_evaluation(0)],
        )

        oof_proba[va_idx] = model.predict(X_oof[va_idx])
        test_probas.append(model.predict(X_test))

        fold_score = balanced_accuracy_score(y_true[va_idx], oof_proba[va_idx].argmax(axis=1))
        fold_scores.append(fold_score)
        print(f"  Fold {fold + 1}: {fold_score:.5f} (best iter: {model.best_iteration})")

    test_proba = np.mean(test_probas, axis=0)
    cv_score = balanced_accuracy_score(y_true, oof_proba.argmax(axis=1))
    print(f"  Overall CV: {cv_score:.5f} (mean fold: {np.mean(fold_scores):.5f})")
    return oof_proba, test_proba, cv_score


print("=== LGB Stacker (all 30 models) ===")
stacker_oof, stacker_test, stacker_cv = lgb_stacker(
    oof_dict, pred_dict, y_train, MODEL_NAMES
)

In [ ]:
# Apply threshold tuning to stacker
stacker_preds, stacker_tuned_cv, stacker_method = apply_best_threshold(
    stacker_oof, stacker_test, y_train
)
print(f"LGB Stacker + {stacker_method}: CV = {stacker_tuned_cv:.5f}")

In [ ]:
# Also try stacker on greedy-selected subset
print("\n=== LGB Stacker (greedy subset) ===")
stacker_gfs_oof, stacker_gfs_test, stacker_gfs_cv = lgb_stacker(
    oof_dict, pred_dict, y_train, selected
)
stacker_gfs_preds, stacker_gfs_tuned_cv, stacker_gfs_method = apply_best_threshold(
    stacker_gfs_oof, stacker_gfs_test, y_train
)
print(f"LGB Stacker (greedy) + {stacker_gfs_method}: CV = {stacker_gfs_tuned_cv:.5f}")

### 6.4 Rank Averaging

In [ ]:
def rank_average(prob_dict, model_names):
    """Convert probabilities to ranks per class, then average across models.
    
    Args:
        prob_dict: Dict mapping model name to probability arrays.
        model_names: List of model names.
    
    Returns:
        Rank-averaged array.
    """
    ranks = []
    for name in model_names:
        proba = prob_dict[name]
        r = np.column_stack([rankdata(proba[:, c]) for c in range(proba.shape[1])])
        ranks.append(r)
    return np.mean(ranks, axis=0)


# Rank avg all models
rank_oof = rank_average(oof_dict, MODEL_NAMES)
rank_test = rank_average(pred_dict, MODEL_NAMES)
rank_preds, rank_cv, rank_method = apply_best_threshold(rank_oof, rank_test, y_train)
print(f"Rank Average (27m) + {rank_method}: CV = {rank_cv:.5f}")

# Rank avg greedy subset
rank_gfs_oof = rank_average(oof_dict, selected)
rank_gfs_test = rank_average(pred_dict, selected)
rank_gfs_preds, rank_gfs_cv, rank_gfs_method = apply_best_threshold(rank_gfs_oof, rank_gfs_test, y_train)
print(f"Rank Average (greedy) + {rank_gfs_method}: CV = {rank_gfs_cv:.5f}")

### 6.5 Equal Weight Average

In [ ]:
# Equal weight all 30 models
eq_oof = np.mean([oof_dict[n] for n in MODEL_NAMES], axis=0)
eq_test = np.mean([pred_dict[n] for n in MODEL_NAMES], axis=0)
eq_preds, eq_cv, eq_method = apply_best_threshold(eq_oof, eq_test, y_train)
print(f"Equal Weight (27m) + {eq_method}: CV = {eq_cv:.5f}")

### 6.6 Logistic Regression Stacker

In [ ]:
def lr_stacker(oof_dict, pred_dict, y_true, model_names, n_splits=5, seed=42):
    """Logistic regression stacker on OOF probabilities.
    
    Args:
        oof_dict: Dict mapping model name to OOF predictions.
        pred_dict: Dict mapping model name to test predictions.
        y_true: Ground truth integer labels.
        model_names: List of model names to use as features.
        n_splits: Number of CV folds.
        seed: Random state.
    
    Returns:
        Tuple of (oof_proba, test_proba, cv_score).
    """
    X_oof = np.hstack([oof_dict[n] for n in model_names])
    X_test = np.hstack([pred_dict[n] for n in model_names])

    n_classes = 3
    oof_proba = np.zeros((len(y_true), n_classes))
    test_probas = []

    skf = StratifiedKFold(n_splits=n_splits, shuffle=True, random_state=seed)

    for fold, (tr_idx, va_idx) in enumerate(skf.split(X_oof, y_true)):
        model = LogisticRegression(
            C=1.0, max_iter=1000, solver="lbfgs",
            multi_class="multinomial", random_state=seed,
        )
        model.fit(X_oof[tr_idx], y_true[tr_idx])
        oof_proba[va_idx] = model.predict_proba(X_oof[va_idx])
        test_probas.append(model.predict_proba(X_test))

    test_proba = np.mean(test_probas, axis=0)
    cv_score = balanced_accuracy_score(y_true, oof_proba.argmax(axis=1))
    return oof_proba, test_proba, cv_score


lr_oof, lr_test, lr_cv = lr_stacker(oof_dict, pred_dict, y_train, MODEL_NAMES)
lr_preds, lr_tuned_cv, lr_method = apply_best_threshold(lr_oof, lr_test, y_train)
print(f"LR Stacker + {lr_method}: CV = {lr_tuned_cv:.5f}")

## 7. Results Summary

In [ ]:
results = {
    f"LGB Stacker 30m + {stacker_method}": (stacker_preds, stacker_tuned_cv, stacker_test),
    f"LGB Stacker greedy + {stacker_gfs_method}": (stacker_gfs_preds, stacker_gfs_tuned_cv, stacker_gfs_test),
    f"Greedy {len(selected)}m + {greedy_method}": (greedy_preds, greedy_cv, greedy_test),
    f"Equal 30m + {eq_method}": (eq_preds, eq_cv, eq_test),
    f"Rank 30m + {rank_method}": (rank_preds, rank_cv, rank_test),
    f"Rank greedy + {rank_gfs_method}": (rank_gfs_preds, rank_gfs_cv, rank_gfs_test),
    f"LR Stacker + {lr_method}": (lr_preds, lr_tuned_cv, lr_test),
}

sorted_results = sorted(results.items(), key=lambda x: -x[1][1])

print(f"{'Method':<45} {'CV (bal_acc)':>12}")
print("=" * 59)
for name, (preds, cv, _) in sorted_results:
    marker = " <-- best" if cv == sorted_results[0][1][1] else ""
    print(f"{name:<45} {cv:.5f}{marker}")

In [ ]:
# Visualize results
method_names = [name for name, _ in sorted_results]
cv_scores = [cv for _, (_, cv, _) in sorted_results]

fig, ax = plt.subplots(figsize=(12, 5))
bars = ax.barh(range(len(method_names)), cv_scores, color="steelblue", edgecolor="black", linewidth=0.5)
bars[0].set_color("#e74c3c")  # highlight best
ax.set_yticks(range(len(method_names)))
ax.set_yticklabels(method_names, fontsize=10)
ax.set_xlabel("CV Balanced Accuracy")
ax.set_title("Ensemble Method Comparison")
ax.set_xlim(min(cv_scores) - 0.001, max(cv_scores) + 0.0005)
for i, v in enumerate(cv_scores):
    ax.text(v + 0.0001, i, f"{v:.5f}", va="center", fontsize=9)
ax.invert_yaxis()
plt.tight_layout()
plt.show()

## 8. Confusion Matrix of Best Method

In [ ]:
best_name, (best_preds_test, best_cv, _) = sorted_results[0]

# Get OOF predictions for confusion matrix
# Use the stacker OOF since it's our best method
if "LGB Stacker 27m" in best_name:
    best_oof_preds = np.argmax(
        np.log(np.clip(stacker_oof, 1e-15, 1.0)) + tune_bias_log(stacker_oof, y_train)[0],
        axis=1,
    ) if "log_bias" in best_name else (stacker_oof * diffevol_thresholds(stacker_oof, y_train)[0]).argmax(axis=1)
else:
    best_oof_preds = stacker_oof.argmax(axis=1)

fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# Absolute counts
cm = confusion_matrix(y_train, best_oof_preds)
ConfusionMatrixDisplay(cm, display_labels=le.classes_).plot(ax=axes[0], cmap="Blues", values_format=",d")
axes[0].set_title(f"{best_name}\nConfusion Matrix (counts)")

# Normalized
cm_norm = confusion_matrix(y_train, best_oof_preds, normalize="true")
ConfusionMatrixDisplay(cm_norm, display_labels=le.classes_).plot(ax=axes[1], cmap="Blues", values_format=".3f")
axes[1].set_title(f"{best_name}\nConfusion Matrix (normalized)")

plt.tight_layout()
plt.show()

# Per-class accuracy
for i, cls in enumerate(le.classes_):
    acc = cm_norm[i, i]
    print(f"  {cls}: {acc:.4f} ({cm[i, i]:,} / {cm[i].sum():,})")
print(f"  Balanced accuracy: {best_cv:.5f}")

## 9. Submission

We use the **LGB stacker** for the final submission rather than the highest-CV method (greedy).
Greedy selection consistently overfits CV: it scores ~0.98084 on CV but only ~0.979 on LB.
The stacker has a much tighter CV-to-LB gap and produces better public LB scores.
This pattern held across all versions (v6 through v11) of our ensemble.

In [ ]:
# Use the LGB stacker (best LB generalization, not highest CV)
# Greedy has higher CV but consistently worse LB due to overfitting
stacker_key = [k for k in results if "LGB Stacker 30m" in k][0]
stacker_preds_final, stacker_cv_final, _ = results[stacker_key]
labels = le.inverse_transform(stacker_preds_final)

submission = pd.DataFrame({
    "id": test["id"],
    "Irrigation_Need": labels,
})

print(f"Submission method: {stacker_key} (CV = {stacker_cv_final:.5f})")
print(f"\nSubmission shape: {submission.shape}")
print(f"\nClass distribution:")
print(submission["Irrigation_Need"].value_counts().reindex(CLASS_ORDER))

submission.to_csv("submission.csv", index=False)
print("\nSaved submission.csv")
submission.head(10)


## 10. How to Reproduce the Full Pipeline

The training code is available in the attached `ps6e4-training-code` dataset. Here is how to train all 30 models from scratch:

### Prerequisites
- AWS account with SageMaker access
- An S3 bucket for data and model artifacts
- IAM role with SageMaker and S3 permissions
- Python environment with `sagemaker`, `boto3`, and `uv` installed locally

### Step 1: Upload data to S3
```bash
aws s3 cp train.csv s3://your-bucket/ensemble-data/train.csv
aws s3 cp test.csv s3://your-bucket/ensemble-data/test.csv
```

### Step 2: Train OTE models (best models)
```python
import sagemaker
from sagemaker.pytorch import PyTorch

# XGBoost + OTE (our breakthrough model, CV 0.97938)
estimator = PyTorch(
    entry_point='train_xgb_ote.py',
    source_dir='./training-code/',
    image_uri='763104351884.dkr.ecr.us-east-1.amazonaws.com/'
              'autogluon-training:1.2-gpu-py311',
    role='your-sagemaker-role-arn',
    instance_count=1,
    instance_type='ml.c5.9xlarge',  # CPU is sufficient
    output_path='s3://your-bucket/output/xgb-ote/',
)
estimator.fit({'training': 's3://your-bucket/ensemble-data/'})
```

Repeat for `train_cat_ote.py` and `train_lgb_ote.py`.

### Step 3: Train v3 pipeline models
The v3 models use a feature store built by `02_feature_engineering.py`. Run that first, then train with `03a_train_xgb.py`, `03b_train_lgb.py`, `03c_train_cat.py`.

### Step 4: Collect predictions
Each training job outputs `model.tar.gz` containing `oof_*.npy` and `pred_*.npy`. Download and extract these.

### Step 5: Run ensemble
Use this notebook with all 30 prediction files. The LGB stacker on all 30 models reproduces the best ensemble CV.

### Key details
- All models use `StratifiedKFold(n_splits=5, shuffle=True, random_state=42)`
- Label encoding is alphabetical: High=0, Low=1, Medium=2
- OOF arrays are shape (630000, 3), test arrays are (270000, 3)
- CPU instances (`ml.c5.9xlarge`) suffice for all GBDTs. Neural networks need GPU (`ml.g4dn.xlarge`)

## Summary

**30 models ensembled:**
- 6 OTE pipeline models (XGBoost x4, CatBoost, LightGBM) with Ordered Target Encoding
- 7 CatBoost v3 models with domain features and Optuna tuning
- 5 LightGBM v3 models
- 6 XGBoost v3 models
- 2 RealMLP neural networks
- 1 TabM neural network
- 3 diversity models (ExtraTrees, Logistic Regression, KNN) with OTE features

**Key findings:**
- Ordered Target Encoding produced the best individual models (CV 0.979+)
- LGB stacking on all 30 models outperforms greedy selection at scale
- More models help stackers but hurt greedy selection
- Weak but algorithmically diverse models (ExtraTrees, LR, KNN) improve the stacker despite low individual CV
- Two complementary feature pipelines (domain-driven v3 and target-driven OTE) provide the diversity that makes stacking effective